# House Stage 1: sensitivity of the frozen ensemble to the training start year

A **training-window robustness analysis** of the frozen Stage 1 ensemble from `01_reg_final_2024_holdout.ipynb`. This is **not model selection.** Everything but the first training year is held fixed:

- **Predictors:** Model A's and Model B's.
- **Incumbency:** Model A's `inc_signed` fixed at +2.5.
- **Estimation:** equal-year WLS (each valid training race weighted ∝ 1 / the number of valid races in its year).
- **Ensemble:** 0.85 · A + 0.15 · B, never re-optimized.

| specification | training years for eval year *t* |
|---|---|
| `start_2006` (baseline) | 2006 … *t* − 2 |
| `start_2008` (exclude 2006) | 2008 … *t* − 2 |
| `start_2010` (exclude 2006 and 2008) | 2010 … *t* − 2 |

**Evaluation:**

- **Eval years:** 2018, 2020, 2022 and 2024, each trained strictly on earlier elections.
- **Matched samples:** within an eval year, every specification scores the same districts (asserted). Only the training rows differ.
- **Subsets:** the frozen notebook's: valid, `competitive_ex_ante` (|lean + generic ballot| ≤ 10, evaluation only) and `competitive_ex_post` (|margin| ≤ 7.5).

**2024 has already been examined.** It was the frozen ensemble's one-shot holdout in `01_reg_final_2024_holdout.ipynb`, so it is **not an untouched holdout for choosing a training start year.** It is reported for completeness, and averages are given both with it (2018–2024) and without it (2018–2022). Neither should be used to pick a start year.

**Short windows are weakly identified:**

- **`start_2010` for 2018** trains on four elections (2010–2016), all under a Democratic president. Model A's national terms (intercept, generic ballot, momentum) then rest on four values, one per year.
- **Model B's** midterm and inflation terms are signed by a president's party that never changes in that window.

Coefficient instability is expected there; the last section measures it.

**Caveat:** `fundraising_advantage`'s historical totals run past Election Day.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown
from patsy import ModelDesc

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from house.experiments import ensemble, evaluation, tracking
from house.experiments.run import run_wls

In [ ]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

## The frozen specification (unchanged from `01_reg_final_2024_holdout.ipynb`)

In [ ]:
TARGET = "adj_margin_2p_norm"
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
EX_ANTE_THRESHOLD = 10
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),  # evaluation only
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),
}
SECTIONS = {"EX_ANTE": "competitive_ex_ante", "EX_POST": "competitive_ex_post", "ALL": "valid"}

FIXED_INC = 2.5         # Model A's imposed incumbency coefficient
WEIGHT_B = 0.15         # the final ensemble: (1 - WEIGHT_B) * A + WEIGHT_B * B
EQUAL_YEAR_WEIGHTS = True

# Known before every election: the White House party (+1 D, -1 R) and whether the election is a midterm
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED),
                               is_midterm=(predictors["year"] % 4 == 2).astype(int))

MODELS = {  # name: (formula, fixed coefficients)
    "A_polling": (f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed"
                  " + gb_momentum_30d_raw", {"inc_signed": FIXED_INC}),
    "B_fundamentals": (f"{TARGET} ~ partisan_lean + fundraising_advantage + pres_party_signed:is_midterm"
                       " + pres_party_signed:cpi_region_log_yoy", {}),
}
A, B = MODELS

# Guards on Model B: no generic ballot, no incumbency / gasoline / college / Hispanic, national terms only signed
_, b_columns = evaluation.formula_variables(MODELS[B][0], predictors.columns)
GB_COLUMNS = ["gen_margin_raw", *(c for c in predictors.columns if c.startswith("gb_"))]
EXCLUDED = ["inc_signed", "log_gas_yoy", "college", "rel_state_college", "rel_college", "hispanic", "rel_state_hispanic"]
assert not set(b_columns) & set(GB_COLUMNS + EXCLUDED), b_columns
b_terms = [term.name() for term in ModelDesc.from_formula(MODELS[B][0]).rhs_termlist]
assert {"pres_party_signed:is_midterm", "pres_party_signed:cpi_region_log_yoy"} <= set(b_terms)
assert not {"pres_party_signed", "is_midterm", "cpi_region_log_yoy"} & set(b_terms), b_terms

SHARED_COLUMNS = evaluation.shared_columns(formulas=[formula for formula, _ in MODELS.values()],
                                           columns=predictors.columns)
for name, (formula, fixed) in MODELS.items():
    print(f"{name:16s} {formula}   fixed: {fixed or 'none'}")
print(f"final = {1 - WEIGHT_B:.2f} * {A} + {WEIGHT_B:.2f} * {B}; shared columns: {SHARED_COLUMNS}")

In [ ]:
def fit(name, train_years, eval_year, use_holdout):
    formula, fixed = MODELS[name]
    return run_wls(
        formula=formula,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        equal_year=EQUAL_YEAR_WEIGHTS,
        fixed=fixed,
        predictors=predictors,
        targets=targets,
        use_holdout=use_holdout,
        require_complete=SHARED_COLUMNS,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=name,
        save=False,
    )


def final_metrics(a, b):
    """(metrics, scored) of the final ensemble of results a and b, scored like a single model."""
    metrics, _, scored = evaluation.evaluate_predictions(ensemble.blend(a, b, WEIGHT_B), targets, predictors, TARGET,
                                                         competitive=COMPETITIVE, race_types=EVAL_RACE_TYPES)
    return metrics, scored


METRICS = ["n", "mae", "rmse", "bias", "accuracy_pct", "precision_D_pct", "precision_R_pct", "missed_races"]
DELTAS = ["mae", "rmse", "accuracy_pct", "missed_races"]


def summary(metrics_by_model):
    """One row per model: METRICS per section, then Δ (model - A) on the same districts."""
    rows = {}
    for name, metrics in metrics_by_model.items():
        row = {}
        for section, subset in SECTIONS.items():
            values, base = metrics.loc[(subset, "all")], metrics_by_model[A].loc[(subset, "all")]
            row |= {(section, m): values[m] for m in METRICS}
            if name != A:
                row |= {(section, f"Δ{m}"): values[m] - base[m] for m in DELTAS}
        rows[name] = row
    return pd.DataFrame.from_dict(rows, orient="index")

## Training windows

Each specification is refit for every eval year on its own expanding window. 2024's window includes HOLDOUT_YEARS, so `use_holdout` is set for that eval year only.

In [ ]:
START_YEARS = {"start_2006": 2006, "start_2008": 2008, "start_2010": 2010}  # specification: first training year
EVAL_YEARS = [2018, 2020, 2022, 2024]
EXAMINED = {2024}  # already scored as the frozen ensemble's holdout: reported, never used to choose


def train_years(start, eval_year):
    return list(range(start, eval_year, 2))


pd.DataFrame({spec: {year: f"{train_years(start, year)[0]}-{train_years(start, year)[-1]} "
                           f"({len(train_years(start, year))} elections)" for year in EVAL_YEARS}
              for spec, start in START_YEARS.items()})

In [ ]:
with warnings.catch_warnings(action="ignore"):  # FEC-missing districts are excluded with a warning, as in every notebook
    fits = {
        spec: {year: {name: fit(name, train_years(start, year), year, use_holdout=year in EXAMINED) for name in MODELS}
               for year in EVAL_YEARS}
        for spec, start in START_YEARS.items()
    }

# matched samples: within a specification and year, A and B share rows; within a year, every specification scores the
# same districts (only the training rows differ)
for year in EVAL_YEARS:
    for spec in START_YEARS:
        runs = fits[spec][year].values()
        assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in runs}) == 1, (spec, year)
    assert len({fits[spec][year][A].counts["eval_rows_hash"] for spec in START_YEARS}) == 1, year

results = {spec: {year: summary({A: fits[spec][year][A].metrics,
                                 "final": final_metrics(fits[spec][year][A], fits[spec][year][B])[0]})
                  for year in EVAL_YEARS} for spec in START_YEARS}
pd.DataFrame({spec: {year: fits[spec][year][A].counts["n_train"] for year in EVAL_YEARS} for spec in START_YEARS}
             ).rename_axis(index="eval_year", columns="training rows")

## Results by year and on average

For each specification and eval year: Model A and the final ensemble, with MAE and winner accuracy per section, and Δ (final − A) on the same districts. ✓ marks an improvement and ✗ a regression.

- `mean 2018-2022` averages the backtest years, each weighted equally.
- `mean 2018-2024` adds the already-examined 2024.

In [ ]:
SHOW = [(section, m) for section in SECTIONS for m in ["mae", "accuracy_pct", "Δmae", "Δaccuracy_pct"]]
AVERAGES = {"mean 2018-2022": [2018, 2020, 2022], "mean 2018-2024": EVAL_YEARS}

rows = {}
for spec in START_YEARS:
    for year in EVAL_YEARS:
        for model in [A, "final"]:
            rows[(spec, year if year not in EXAMINED else f"{year} (examined)", model)] = results[spec][year].loc[model]
    for label, years in AVERAGES.items():
        for model in [A, "final"]:
            rows[(spec, label, model)] = sum(results[spec][year].loc[model] for year in years) / len(years)
by_year = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["spec", "eval_year", "model"])[SHOW]
tracking.style_best_of(by_year)

## Against the 2006 baseline

Each later start year minus `start_2006`, for the same model on the same districts. A negative ΔMAE and a positive Δaccuracy mean that excluding older elections helped. `years better` counts the backtest years (2018–2022) where MAE improved; 2024 is shown but not counted.

In [ ]:
BASE = "start_2006"
KEY_METRICS = [(section, m) for section in SECTIONS for m in ["mae", "accuracy_pct"]]

rows = {}
for spec in START_YEARS:
    if spec == BASE:
        continue
    for model in [A, "final"]:
        row = {}
        for section, metric in KEY_METRICS:
            diffs = {year: results[spec][year].loc[model, (section, metric)] - results[BASE][year].loc[model, (section, metric)]
                     for year in EVAL_YEARS}
            for year, diff in diffs.items():
                row[(f"{section} {metric}", str(year) + (" (examined)" if year in EXAMINED else ""))] = diff
            backtest = [diffs[year] for year in [2018, 2020, 2022]]
            row[(f"{section} {metric}", "mean 2018-2022")] = sum(backtest) / 3
            if metric == "mae":
                row[(f"{section} {metric}", "years better")] = sum(d < 0 for d in backtest)
        rows[(spec, model)] = row
vs_base = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["spec", "model"])


def color(v, col):
    if pd.isna(v) or col[1] == "years better" or v == 0:
        return ""
    better = v < 0 if col[0].endswith("mae") else v > 0
    return f"color: {tracking.GOOD if better else tracking.BAD}"


vs_base.style.format({c: "{:.0f}/3" if c[1] == "years better" else "{:+.2f}" for c in vs_base.columns}).apply(
    lambda col: [color(v, col.name) for v in col])

## Coefficient stability

The key coefficients in every training window:

- **Model A:** generic ballot, momentum, partisan lean and lean × incumbency. Incumbency is fixed at 2.5.
- **Model B:** fundraising, the midterm penalty, inflation and partisan lean.

The summary table gives each specification's **range** (max − min across its four windows) and **standard deviation** across windows. Larger values mean less stable estimates.

In [ ]:
KEY_TERMS = {
    A: ["gen_margin_raw", "gb_momentum_30d_raw", "partisan_lean", "partisan_lean:inc_signed", "Intercept"],
    B: ["fundraising_advantage", "pres_party_signed:is_midterm", "pres_party_signed:cpi_region_log_yoy",
        "partisan_lean", "Intercept"],
}
coefficients = pd.DataFrame({
    (spec, year): {(model, term): fits[spec][year][model].coefficients.loc[term, "coef"]
                   for model, terms in KEY_TERMS.items() for term in terms}
    for spec in START_YEARS for year in EVAL_YEARS
}).T.rename_axis(["spec", "eval_year"])
display(Markdown("### Coefficients by training window"))
display(coefficients.style.format(precision=3))

stability = pd.concat({
    "range": coefficients.groupby(level="spec", sort=False).agg(lambda c: c.max() - c.min()),
    "std": coefficients.groupby(level="spec", sort=False).std(),
}, names=["stat", "spec"])
display(Markdown("### Stability across each specification's four windows (larger = less stable)"))
stability.style.format(precision=3).background_gradient(axis=0, cmap="Reds")

In [ ]:
# standard errors of the key coefficients: a shorter window estimates the national terms from fewer elections
std_errors = pd.DataFrame({
    (spec, year): {(model, term): fits[spec][year][model].coefficients.loc[term, "std_err"]
                   for model, terms in KEY_TERMS.items() for term in terms if term != "Intercept"}
    for spec in START_YEARS for year in EVAL_YEARS
}).T.rename_axis(["spec", "eval_year"])
display(Markdown("### Standard errors by training window (OLS-style; they ignore within-year correlation, so they "
                 "understate the uncertainty of the national terms)"))
std_errors.style.format(precision=3)